# FitQSD
Explore different formulations to fit the distribution generated by the POVM
derived by the noiseless problem.

In [18]:
import logging
logging.basicConfig(
    filename=f"dev.log",
    filemode="a",
    format="{asctime} {levelname} {filename}:{lineno}: {message}",
    datefmt="%Y-%m-%d %H:%M:%S",
    style="{",
    level=logging.INFO,  # Qiskit dumps too many DEBUG messages
    encoding="utf-8",
)
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib.font_manager').disabled = True
logging.getLogger('PIL.PngImagePlugin').disabled = True
logging.getLogger('matplotlib.mathtext').disabled = True
logger = logging.getLogger(__name__)

In [19]:
import warnings
import mosek

# Suppress the specific warning
warnings.filterwarnings("ignore", 
                       message="Argument sub in putvarboundlist: Incorrect array format causing data to be copied",
                       category=UserWarning,
                       module="mosek")

In [20]:
import sys
sys.path.append("../")

In [21]:
import numpy as np
import cvxpy as cp

In [22]:
from flow.problem_spec import *
from utils.handy_states import *

from flow.solve_mix import *
from utils.inner_product import *
from scipy.spatial.distance import jensenshannon

## Define the input states

In [23]:
state_dict = sv_simple_2(0.2, 0.5, 0.7)
num_qubits = state_dict["num_qubits"]
num_states = state_dict["num_states"]
state_vec = state_dict["states"]
dense_mat = [DensityMatrix(_) for _ in state_vec]

In [24]:
"""
num_qubits = 3
num_states = 3
state_vec, dense_mat = sv_coh_asymm_small(num_qubits=num_qubits)
"""

'\nnum_qubits = 3\nnum_states = 3\nstate_vec, dense_mat = sv_coh_asymm_small(num_qubits=num_qubits)\n'

In [25]:
"""
num_qubits = 1
num_states = 2

state_vec = [
    Statevector([1, 0]),
    Statevector([1/np.sqrt(2), 1/np.sqrt(2)]),
]
dense_mat = [DensityMatrix(_) for _ in state_vec]
"""

'\nnum_qubits = 1\nnum_states = 2\n\nstate_vec = [\n    Statevector([1, 0]),\n    Statevector([1/np.sqrt(2), 1/np.sqrt(2)]),\n]\ndense_mat = [DensityMatrix(_) for _ in state_vec]\n'

In [26]:
"""
num_qubits = 6
num_states = 3

state_vec, _ = sv_coh_asymm_small(num_qubits=num_qubits)
dense_mat = [DensityMatrix(_) for _ in state_vec]
"""

'\nnum_qubits = 6\nnum_states = 3\n\nstate_vec, _ = sv_coh_asymm_small(num_qubits=num_qubits)\ndense_mat = [DensityMatrix(_) for _ in state_vec]\n'

In [27]:
qsd_problem = ProblemSpec(
    num_qubits=num_qubits,
    num_states=num_states,
    case_id="0809_test",
    state_type="densitymatrix",
)

In [28]:
noise_levels = [0.1 ** (6 - 0.5 * i) for i in range(11)]
disturbance_states = [
    DensityMatrix(
        ProblemSpec.depolarizing_noise_channel(num_qubits=num_qubits)
    )
    for _ in range(num_states)
]

## Compute the noiseless distribution

In [ ]:
n = qsd_problem.num_states
# Comment/uncomment to switch between different prior probabilities
prior_prob = np.ones(n) * (1 / n)
# prior_prob = [1/9, 3/9, 5/9]
qsd_problem.prior_prob = prior_prob

In [30]:
qsd_problem.set_states(
    state_type="statevector",
    states=state_vec,
    overwrite=True,
)
ideal_result = apply_Eldar(
    problem_spec=qsd_problem,
    prior_prob=prior_prob,
    cvxpy_settings={
        "solver": cp.MOSEK,
        "verbose": False,
        "eps": 1e-8,
    }
)

ideal_distrib = calculate_prob_matrix_simple(
    prior_probs=prior_prob,
    povm=ideal_result["povm"],
    states=dense_mat,
)
print(ideal_result["p_succ"])

0.7571478026188229


/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18925: UserWarning: Argument subj in putclist: Incorrect array format causing data to be copied
  warnings.warn("Argument subj in putclist: Incorrect array format causing data to be copied");
/home/ChienKaiMa/QSD/.venv/lib/python3.12/site-packages/mosek/__init__.py:18349: UserWarning: Argument sub in putconboundlist: Incorrect array format causing data to be copied
  warnings.warn("Argument sub in putconboundlist: Incorrect array format causing data to be copied");


## Use the POVM in noisy states

In [31]:
def report_ideal(ideal_distrib):
    print("Ideal input states")
    print("POVM for ideal optimal UQSD")
    print(ideal_distrib)
    print()


def dist_1(m):
    correct_results = [m[i][i] for i in range(len(m))]
    p_succ_ni = sum(correct_results)
    return np.concatenate((correct_results, np.array([1.0 - p_succ_ni])))


def dist_2(m):
    measured_results = [
        sum([m[j][i] for j in range(len(m))]) for i in range(len(m))
    ]
    p_succ_ni = sum(measured_results)
    return np.concatenate((measured_results, np.array([1.0 - p_succ_ni])))


def dist_3(m):
    """Yeah. No changes.
    However, we have to calculate the ideal distribution in the same way.
    """
    return np.array(m).flatten()


def get_sqrt_dist(a, b):
    return np.sqrt(sum([abs(a[i] - b[i]) ** 2 for i in range(len(a))]))


def report_noisy_states_with_ideal_povm(
    noise_level,
    ideal_distrib,
    qsd_problem: ProblemSpec,
    jsd_result,
    psucc_result,
):
    combined_states = [
        (1 - noise_level) * dense_mat[_]
        + noise_level * disturbance_states[_].data
        for _ in range(num_states)
    ]

    qsd_problem.set_states(
        state_type="densitymatrix",
        states=combined_states,
        overwrite=True,
    )
    # m is prob_matrix
    m = calculate_prob_matrix_simple(
        qsd_problem.prior_prob, ideal_result["povm"], combined_states
    )
    # for i in m:
    #     print(i)
    noisy_distrib_with_ideal_povm = dist_3(m)
    print(f"Noisy input states with depolarizing noise = {noise_level:3g}")
    print("POVM for ideal optimal UQSD")
    print(noisy_distrib_with_ideal_povm)
    np.set_printoptions(precision=4)
    psucc = 0
    for i in range(qsd_problem.num_states):
        psucc += m[i][i]
    print(psucc)

    print("alpha", np.array(calculate_errors(m)[0]))
    print("beta ", np.array(calculate_errors(m)[1]))
    js_dist = jensenshannon(
        np.array(ideal_distrib).flatten(), noisy_distrib_with_ideal_povm
    )
    psucc_result.append(psucc)
    jsd_result.append(js_dist)
    print(
        f"Jensen-Shannon dist {np.format_float_scientific(js_dist, precision=4)}"
    )
    sqrt_dist = get_sqrt_dist(
        np.array(ideal_distrib).flatten(), noisy_distrib_with_ideal_povm
    )
    print(
        f"Root sum of squares {np.format_float_scientific(sqrt_dist, precision=4)}"
    )
    print()

In [32]:
jsd_result = []
sqrtd_result = []
psucc_result = []
for noise_level in noise_levels:
    report_noisy_states_with_ideal_povm(
        noise_level=noise_level,
        ideal_distrib=ideal_distrib,
        qsd_problem=qsd_problem,
        jsd_result=jsd_result,
        psucc_result=psucc_result,
    )

Noisy input states with depolarizing noise = 1e-06
POVM for ideal optimal UQSD
[1.0005e-01 2.1475e-08 2.2528e-08 1.1063e-02 7.6284e-08 2.3985e-01
 6.7584e-08 9.3482e-02 1.2714e-07 1.0738e-07 4.1725e-01 1.3831e-01]
0.757147247964814
alpha [4.3982e-07 5.9982e-07 5.6206e-07]
beta  [2.0333e-06 5.3722e-07 2.1597e-07]
Jensen-Shannon dist 3.8261e-04
Root sum of squares 4.1836e-07

Noisy input states with depolarizing noise = 3.16228e-06
POVM for ideal optimal UQSD
[1.0005e-01 6.7911e-08 7.1240e-08 1.1063e-02 2.4123e-07 2.3985e-01
 2.1372e-07 9.3482e-02 4.0205e-07 3.3956e-07 4.1725e-01 1.3831e-01]
0.7571460486488416
alpha [1.3908e-06 1.8968e-06 1.7774e-06]
beta  [6.4297e-06 1.6988e-06 6.8295e-07]
Jensen-Shannon dist 6.8038e-04
Root sum of squares 1.323e-06

Noisy input states with depolarizing noise = 1e-05
POVM for ideal optimal UQSD
[1.0005e-01 2.1475e-07 2.2528e-07 1.1063e-02 7.6284e-07 2.3985e-01
 6.7584e-07 9.3483e-02 1.2714e-06 1.0738e-06 4.1725e-01 1.3831e-01]
0.7571422560787344
alpha [

## New formulations

In [33]:
def run_qsd(
    qsd_problem: ProblemSpec,
    cvxpy_problem,
    jsd_result,
    sqrtd_result,
    psucc_result,
    eps=1e-8,
):
    cvxpy_settings = {
        "solver": cp.MOSEK,
        "verbose": False,
        # "verbose": True,
        # "requires_grad": True,
        # "mkl": True,
        "eps": eps,
        # "acceleration_lookback": 10,
        # "warm_start": True,
    }
    cvxpy_problem.solve(**cvxpy_settings)

    # print(cvxpy_problem.status)
    # print(cvxpy_problem.solution.opt_val)
    vars = cvxpy_problem.variables()

    fitqd_povm = [var.value for var in vars]
    # print(fitqd_povm)

    prob_mat = calculate_prob_matrix_simple(
        prior_probs=prior_prob,
        povm=fitqd_povm,
        states=qsd_problem.states,
    )
    # print(prob_mat)

    psucc = 0
    for i in range(qsd_problem.num_states):
        psucc += prob_mat[i][i]
    psucc_result.append(psucc)

    js_dist = jensenshannon(
        np.array(ideal_distrib).flatten(),
        np.array(prob_mat).flatten(),
    )
    jsd_result.append(js_dist)

    sqrt_dist = get_sqrt_dist(
        np.array(ideal_distrib).flatten(),
        np.array(prob_mat).flatten(),
    )
    sqrtd_result.append(sqrt_dist)
    #
    # print("alpha", np.array(calculate_errors(prob_mat)[0]))
    # print("beta ", max(calculate_errors(prob_mat)[1]))
    #
    # print()

In [34]:
methods = [max_psucc_min_diff_problem, min_l1_problem, min_ss_problem]
eps_list = [1e-8, 1e-9]
for eps in eps_list:
    print("eps =", eps)
    for method in methods:
        print(f"\n{method.__name__}")
        jsd_result = []
        sqrtd_result = []
        psucc_result = []
        for noise_level in noise_levels:
            combined_states = [
                (1 - noise_level) * dense_mat[_]
                + noise_level * disturbance_states[_].data
                for _ in range(num_states)
            ]
            qsd_problem.set_states(
                state_type="densitymatrix",
                states=combined_states,
                overwrite=True,
            )
            # print("Noise level", np.format_float_scientific(noise_level, precision=4))
            cvxpy_problem = method(
                ideal_distrib=ideal_distrib,
                qsd_problem=qsd_problem,
                prior_prob=prior_prob,
            )
            run_qsd(
                qsd_problem=qsd_problem,
                cvxpy_problem=cvxpy_problem,
                jsd_result=jsd_result,
                sqrtd_result=sqrtd_result,
                psucc_result=psucc_result,
                eps=eps,
            )
        for item in jsd_result:
            print(item)
        print()
        for item in sqrtd_result:
            print(item)
        print()
        for item in psucc_result:
            print(item)

eps = 1e-08

max_psucc_min_diff_problem


0.00038588224300852475
0.0006867069844352426
0.0012143269388068404
0.002153145421842637
0.003827186184119947
0.006804527285777254
0.012102016089930724
0.021529602643468065
0.03833928862135809
0.0684837951940669
0.12357179224071624

3.4466831681850293e-07
1.0889663834034922e-06
3.4280946286797954e-06
1.0808946898381577e-05
3.4168907959573875e-05
0.00010803134464641383
0.00034167532041313976
0.0010809342646582246
0.0034232242945770003
0.010875935996500628
0.03491753343644979

0.7571473741331981
0.7571464473501404
0.7571435534836222
0.7571344300764425
0.7571055442560464
0.7570142151179708
0.7567253034911556
0.7558112158009058
0.7529151914585249
0.7437028196286228
0.7140084532758364

min_l1_problem
0.0003899531813478359
0.0006835784309886623
0.0012111109352413735
0.0021523087001328904
0.0038266368909695004
0.006804580514176393
0.0121018828128721
0.021529617389199014
0.03833930606439573
0.06848374127461737
0.12357179268789277

2.877610272044653e-07
8.889967351210513e-07
2.794738012977822e-0